# STEA on Wikidata12K Cross-Lingual Temporal Entity Alignment

PyTorch Colab notebook for **STEA** (Cai et al., COLING 2022) on the **Wikidata12K cross-lingual** dataset.

**Dataset pairs:** `en_fr`, `en_fa`, `en_zh`, `en_ja`

Each pair folder contains:
- `quadruples_en.tsv` / `quadruples_<lang>.tsv` — head_id, rel_id, tail_id, begin, end
- `alignment_all.tsv` — gold entity pairs
- `entity2id_*.tsv`, `relation2id.tsv`

**STEA defaults (paper):** dim=100, depth=2, margin=3, dropout=0.3, α=0.3, 5 iterations × 1200 epochs, CSLS.

### Colab setup
1. Runtime → Change runtime type → **GPU**
2. Upload `wikidata12k_crosslingual.zip` to Drive (or use the path below)
3. Set `LANG_PAIR` (e.g. `en_fr`)
4. Run all cells top to bottom

In [1]:
# ============================================================
# 1. Imports and reproducibility
# ============================================================

import os
import re
import zipfile
import random
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: enable a Colab GPU runtime for faster training.")

PyTorch: 2.11.0+cu128
Device: cuda
GPU: Tesla T4


In [2]:
# ============================================================
# 2. Mount Google Drive
# ============================================================

from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# ============================================================
# 3. Configuration
# ============================================================

# Language pair: en_fr | en_fa | en_zh | en_ja
LANG_PAIR = "en_fa"

# Where the zip lives on Drive (or already extracted folder)
ZIP_PATH = Path("/content/drive/MyDrive/wikidata12k_crosslingual.zip")
EXTRACT_ROOT = Path("/content/wikidata12k_crosslingual")

# STEA hyperparameters (paper defaults)
TRAIN_SEED = 200          # number of supervised alignment seeds
DIM = 100
DEPTH = 2
MARGIN = 3.0
DROPOUT = 0.3
ALPHA = 0.3               # weight of temporal similarity
LR = 0.005
EPOCHS_PER_ITER = 1200
ITERATIONS = 5
USE_ITERATIVE_STEA = True

CSLS_K = 10
TOP_K = (1, 5, 10)

print("LANG_PAIR:", LANG_PAIR)
print("DEVICE:", DEVICE)

LANG_PAIR: en_fa
DEVICE: cuda


In [4]:
# ============================================================
# 4. Unpack dataset and locate language-pair folder
# ============================================================

if not EXTRACT_ROOT.exists():
    if ZIP_PATH.exists():
        print("Extracting", ZIP_PATH)
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall("/content")
    else:
        # Fallback: look for zip in /content
        candidates = list(Path("/content").glob("**/wikidata12k_crosslingual.zip"))
        if candidates:
            with zipfile.ZipFile(candidates[0], "r") as zf:
                zf.extractall("/content")
        else:
            raise FileNotFoundError(
                f"Could not find dataset. Put the zip at {ZIP_PATH} or upload it to /content."
            )

# Folder may be /content/wikidata12k_crosslingual or nested
if (EXTRACT_ROOT / LANG_PAIR).is_dir():
    PAIR_DIR = EXTRACT_ROOT / LANG_PAIR
elif (Path("/content") / "wikidata12k_crosslingual" / LANG_PAIR).is_dir():
    PAIR_DIR = Path("/content") / "wikidata12k_crosslingual" / LANG_PAIR
else:
    found = list(Path("/content").rglob(LANG_PAIR))
    found = [p for p in found if p.is_dir() and (p / "alignment_all.tsv").exists()]
    if not found:
        raise FileNotFoundError(f"Language pair folder not found: {LANG_PAIR}")
    PAIR_DIR = found[0]

print("PAIR_DIR:", PAIR_DIR)
for p in sorted(PAIR_DIR.iterdir()):
    print(f"  {p.name:24s} {p.stat().st_size/1024:.1f} KB")

PAIR_DIR: /content/wikidata12k_crosslingual/en_fa
  alignment_all.tsv        416.1 KB
  entity2id_en.tsv         417.0 KB
  entity2id_fa.tsv         273.6 KB
  quadruples_en.tsv        1412.4 KB
  quadruples_fa.tsv        694.8 KB
  relation2id.tsv          0.6 KB


In [5]:
# ============================================================
# 5. Load Wikidata12K pair (quadruples + alignment)
# ============================================================

def parse_time_token(tok: str) -> int:
    """
    Convert Wikidata-style timestamps to integer year IDs.
    Examples: '1926-##-##' -> 1926, '2008-##-##' -> 2008, '####-##-##' -> 0
    """
    tok = tok.strip()
    m = re.match(r"^(-?\d+)", tok)
    if m:
        return int(m.group(1))
    return 0  # unknown / missing year


def load_quadruples(path):
    """TSV with header: head_id rel_id tail_id begin end"""
    triples = []
    entities = set()
    relations = set()
    times = set()
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 5:
                parts = line.split()
            if len(parts) < 5:
                continue
            h, r, t = int(parts[0]), int(parts[1]), int(parts[2])
            ts = parse_time_token(parts[3])
            te = parse_time_token(parts[4])
            triples.append((h, r, t, ts, te))
            entities.update((h, t))
            relations.add(r)
            times.update((ts, te))
    return triples, entities, relations, times


def load_alignment(path):
    """TSV with header: id_en id_<lang> ..."""
    pairs = []
    with open(path, "r", encoding="utf-8") as f:
        header = f.readline()
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 2:
                continue
            pairs.append((int(parts[0]), int(parts[1])))
    return pairs


# Detect language suffix from LANG_PAIR (e.g. en_fr -> fr)
lang_a, lang_b = LANG_PAIR.split("_")
quad_a_path = PAIR_DIR / f"quadruples_{lang_a}.tsv"
quad_b_path = PAIR_DIR / f"quadruples_{lang_b}.tsv"
align_path = PAIR_DIR / "alignment_all.tsv"

for p in [quad_a_path, quad_b_path, align_path]:
    if not p.exists():
        raise FileNotFoundError(p)

triples1, e1_raw, r1_raw, t1_raw = load_quadruples(quad_a_path)
triples2, e2_raw, r2_raw, t2_raw = load_quadruples(quad_b_path)
all_pairs = load_alignment(align_path)

print(f"KG1 ({lang_a})")
print(f"  triples: {len(triples1)}")
print(f"  entities: {len(e1_raw)}  id range: ({min(e1_raw)}, {max(e1_raw)})")
print(f"  relations: {len(r1_raw)}")
print(f"  times (years): {len(t1_raw)}")

print(f"\nKG2 ({lang_b})")
print(f"  triples: {len(triples2)}")
print(f"  entities: {len(e2_raw)}  id range: ({min(e2_raw)}, {max(e2_raw)})")
print(f"  relations: {len(r2_raw)}")
print(f"  times (years): {len(t2_raw)}")

print(f"\nAlignment pairs: {len(all_pairs)}")
print(f"Raw entity-ID overlap between KGs: {len(e1_raw & e2_raw)}")

KG1 (en)
  triples: 40446
  entities: 12502  id range: (0, 12501)
  relations: 24
  times (years): 618

KG2 (fa)
  triples: 20165
  entities: 6409  id range: (0, 6408)
  relations: 23
  times (years): 451

Alignment pairs: 6389
Raw entity-ID overlap between KGs: 6409


In [6]:
# ============================================================
# 6. Remap IDs (separate entity namespaces) + train/test split
# ============================================================

def make_maps(id_set):
    sorted_ids = sorted(id_set)
    return {x: i for i, x in enumerate(sorted_ids)}


e1_map = make_maps(e1_raw)
e2_map = make_maps(e2_raw)

# Relations: shared vocabulary across languages in this dataset
all_rels = r1_raw | r2_raw
r_map = make_maps(all_rels)

N1 = len(e1_map)
N2 = len(e2_map)
NUM_ENTITIES = N1 + N2
NUM_RELATIONS = len(r_map)


def remap_triples(triples, e_map, r_map, entity_offset=0):
    out = []
    for h, r, t, ts, te in triples:
        out.append((
            e_map[h] + entity_offset,
            r_map[r],
            e_map[t] + entity_offset,
            ts,
            te,
        ))
    return out


triples1_m = remap_triples(triples1, e1_map, r_map, 0)
triples2_m = remap_triples(triples2, e2_map, r_map, N1)

# Filter alignment to entities that appear in both quadruple graphs
valid_pairs = []
for a, b in all_pairs:
    if a in e1_map and b in e2_map:
        valid_pairs.append((e1_map[a], e2_map[b] + N1))

valid_pairs = np.asarray(valid_pairs, dtype=np.int64)
np.random.shuffle(valid_pairs)

if len(valid_pairs) < TRAIN_SEED:
    raise ValueError(f"Only {len(valid_pairs)} pairs, but TRAIN_SEED={TRAIN_SEED}")

seed_pairs = valid_pairs[:TRAIN_SEED].copy()
eval_pairs = valid_pairs[TRAIN_SEED:].copy()

print(f"KG1 entities (N1): {N1}")
print(f"KG2 entities (N2): {N2}")
print(f"Total entities: {NUM_ENTITIES}")
print(f"Relations: {NUM_RELATIONS}")
print(f"Valid alignment pairs: {len(valid_pairs)}")
print(f"Seed pairs: {len(seed_pairs)}")
print(f"Eval pairs: {len(eval_pairs)}")

KG1 entities (N1): 12502
KG2 entities (N2): 6409
Total entities: 18911
Relations: 24
Valid alignment pairs: 6389
Seed pairs: 200
Eval pairs: 6189


In [7]:
# ============================================================
# 7. Build graph, relation incidence, time dictionaries
# ============================================================

neighbors = [set() for _ in range(NUM_ENTITIES)]
entity_relations = [set() for _ in range(NUM_ENTITIES)]
time_dict = [set() for _ in range(NUM_ENTITIES)]

for h, r, t, ts, te in triples1_m + triples2_m:
    neighbors[h].add(t)
    neighbors[t].add(h)
    entity_relations[h].add(r)
    entity_relations[t].add(r)
    # STEA time dict: point or interval endpoints
    time_dict[h].update((ts, te))
    time_dict[t].update((ts, te))

# Sparse adjacency with self-loops, row-normalized (D^{-1} A)
rows, cols = [], []
for i in range(NUM_ENTITIES):
    rows.append(i); cols.append(i)
    for j in neighbors[i]:
        rows.append(i); cols.append(j)

rows = np.asarray(rows, dtype=np.int64)
cols = np.asarray(cols, dtype=np.int64)
degree = np.bincount(rows, minlength=NUM_ENTITIES).astype(np.float32)
values = 1.0 / degree[rows]

adj = torch.sparse_coo_tensor(
    torch.tensor(np.vstack([rows, cols]), dtype=torch.long),
    torch.tensor(values, dtype=torch.float32),
    size=(NUM_ENTITIES, NUM_ENTITIES),
).coalesce().to(DEVICE)

# Entity–relation incidence (for relation feature aggregation)
er_rows, er_cols = [], []
for eid, rels in enumerate(entity_relations):
    for rid in rels:
        er_rows.append(eid)
        er_cols.append(rid)

er_indices = torch.tensor(np.vstack([er_rows, er_cols]), dtype=torch.long)
relation_incidence = torch.sparse_coo_tensor(
    er_indices,
    torch.ones(len(er_rows), dtype=torch.float32),
    size=(NUM_ENTITIES, NUM_RELATIONS),
).coalesce()

relation_count = torch.zeros(NUM_ENTITIES, 1, dtype=torch.float32)
for eid, rels in enumerate(entity_relations):
    relation_count[eid, 0] = max(len(rels), 1)

print("Adjacency nnz:", adj._nnz())
print("Avg degree (incl. self):", float(degree.mean()))
print("Relation incidence nnz:", relation_incidence._nnz())
print("Entities with empty time dict:", sum(1 for s in time_dict if len(s) == 0))

/tmp/ipykernel_9121/1659807966.py:30: UserWarning: Sparse invariant checks are implicitly disabled. Memory errors (e.g. SEGFAULT) will occur when operating on a sparse tensor which violates the invariants, but checks incur performance overhead. To silence this warning, explicitly opt in or out. See `torch.sparse.check_sparse_tensor_invariants.__doc__` for guidance.  (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:760.)
  adj = torch.sparse_coo_tensor(


Adjacency nnz: 114897
Avg degree (incl. self): 6.075776100158691
Relation incidence nnz: 24058
Entities with empty time dict: 0


In [8]:
# ============================================================
# 8. STEA model + loss
# ============================================================

class STEA(nn.Module):
    def __init__(self, num_entities, num_relations, dim=100, depth=2, dropout=0.3):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.dim = dim
        self.depth = depth

        self.entity_emb = nn.Parameter(torch.empty(num_entities, dim))
        self.relation_emb = nn.Parameter(torch.empty(num_relations, dim))
        nn.init.xavier_uniform_(self.entity_emb)
        nn.init.xavier_uniform_(self.relation_emb)
        self.dropout = nn.Dropout(dropout)

    def forward(self, adj):
        # Eq.1 structural neighborhood average (via D^{-1}A)
        structural = torch.sparse.mm(adj, self.entity_emb)

        # Eq.2 average incident relation embeddings
        relational = torch.sparse.mm(
            relation_incidence.to(self.entity_emb.device),
            self.relation_emb,
        )
        relational = relational / relation_count.to(self.entity_emb.device).clamp_min(1.0)

        # Eq.3 feature fusion
        h = torch.cat([structural, relational], dim=1)

        # Eq.5 multi-layer global-aware representation
        representations = [h]
        for _ in range(self.depth - 1):
            h = torch.sparse.mm(adj, h)
            h = F.relu(h)
            h = self.dropout(h)
            representations.append(h)

        return torch.cat(representations, dim=1)


def stea_triplet_loss(emb, pairs, margin=3.0):
    """Manhattan-distance margin loss (Eq. 6–7)."""
    if len(pairs) == 0:
        return emb.sum() * 0.0

    device = emb.device
    p = torch.as_tensor(pairs, dtype=torch.long, device=device)
    src, tgt = p[:, 0], p[:, 1]

    corrupt_source = torch.rand(len(p), device=device) < 0.5
    neg_src, neg_tgt = src.clone(), tgt.clone()
    rand_src = torch.randint(0, N1, (len(p),), device=device)
    rand_tgt = torch.randint(N1, N1 + N2, (len(p),), device=device)
    neg_src[corrupt_source] = rand_src[corrupt_source]
    neg_tgt[~corrupt_source] = rand_tgt[~corrupt_source]

    pos_dist = torch.abs(emb[src] - emb[tgt]).sum(dim=1)
    neg_dist = torch.abs(emb[neg_src] - emb[neg_tgt]).sum(dim=1)
    return F.relu(pos_dist - neg_dist + margin).mean()


model = STEA(NUM_ENTITIES, NUM_RELATIONS, dim=DIM, depth=DEPTH, dropout=DROPOUT).to(DEVICE)
optimizer = torch.optim.RMSprop(model.parameters(), lr=LR)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters() if p.requires_grad))

STEA(
  (dropout): Dropout(p=0.3, inplace=False)
)
Parameters: 1893500


In [9]:
# ============================================================
# 9. Temporal similarity matrix (STEA Eq. 8)
# ============================================================

def temporal_similarity_matrix(n1, n2, time_dicts):
    """st(i,j) = 2*c / (m+n) where c = |T_i ∩ T_j|."""
    S = np.zeros((n1, n2), dtype=np.float32)
    target_sets = [time_dicts[n1 + j] for j in range(n2)]
    for i in range(n1):
        s = time_dicts[i]
        m = len(s)
        if m == 0:
            continue
        for j, tset in enumerate(target_sets):
            n = len(tset)
            if n == 0:
                continue
            c = len(s & tset)
            if c:
                S[i, j] = (2.0 * c) / (m + n)
    return S


print(f"Building temporal similarity matrix ({N1} x {N2})...")
time_sim = temporal_similarity_matrix(N1, N2, time_dict)
print("min/max:", float(time_sim.min()), float(time_sim.max()))
print("fraction exact match (1.0):", float(np.mean(time_sim == 1.0)))
print("fraction nonzero:", float(np.mean(time_sim > 0)))

Building temporal similarity matrix (12502 x 6409)...
min/max: 0.0 1.0
fraction exact match (1.0): 0.001398521750640665
fraction nonzero: 0.38056145998696694


In [10]:
# ============================================================
# 10. CSLS + evaluation + mutual-NN bootstrap
# ============================================================

def csls_similarity(similarity, k=10):
    k = min(k, similarity.shape[0], similarity.shape[1])
    src_r = np.mean(np.partition(similarity, -k, axis=1)[:, -k:], axis=1)
    tgt_r = np.mean(np.partition(similarity, -k, axis=0)[-k:, :], axis=0)
    return 2.0 * similarity - src_r[:, None] - tgt_r[None, :]


def evaluate_embeddings(emb, eval_pairs, alpha=0.3, use_csls=True, csls_k=10):
    source = F.normalize(emb[:N1], p=2, dim=1)
    target = F.normalize(emb[N1:], p=2, dim=1)
    emb_sim = (source @ target.T).detach().cpu().numpy()

    if use_csls:
        emb_sim = csls_similarity(emb_sim, csls_k)

    final_sim = (1.0 - alpha) * emb_sim + alpha * time_sim

    gt = {int(a): int(b) - N1 for a, b in eval_pairs}
    ranks_l2r = []
    for src, tgt in gt.items():
        order = np.argsort(-final_sim[src])
        rank = int(np.where(order == tgt)[0][0]) + 1
        ranks_l2r.append(rank)

    gt_r = {int(b) - N1: int(a) for a, b in eval_pairs}
    ranks_r2l = []
    for tgt, src in gt_r.items():
        order = np.argsort(-final_sim[:, tgt])
        rank = int(np.where(order == src)[0][0]) + 1
        ranks_r2l.append(rank)

    def metrics(ranks):
        ranks = np.asarray(ranks)
        return {
            "Hits@1": float(np.mean(ranks <= 1)),
            "Hits@5": float(np.mean(ranks <= 5)),
            "Hits@10": float(np.mean(ranks <= 10)),
            "MRR": float(np.mean(1.0 / ranks)),
        }

    return {
        "left_to_right": metrics(ranks_l2r),
        "right_to_left": metrics(ranks_r2l),
        "similarity": final_sim,
    }


def print_metrics(result, title="Evaluation"):
    print("\n" + title)
    print("-" * len(title))
    for direction in ["left_to_right", "right_to_left"]:
        print(direction)
        for k, v in result[direction].items():
            if k.startswith("Hits"):
                print(f"  {k:8s}: {v*100:.2f}%")
            else:
                print(f"  {k:8s}: {v:.4f}")


def mutual_nearest_pairs(similarity, existing_pairs=None):
    src_best = np.argmax(similarity, axis=1)
    tgt_best = np.argmax(similarity, axis=0)
    existing = set()
    if existing_pairs is not None:
        existing = {(int(a), int(b)) for a, b in existing_pairs}
    new_pairs = []
    for i, j in enumerate(src_best):
        if tgt_best[j] == i:
            pair = (i, N1 + int(j))
            if pair not in existing:
                new_pairs.append(pair)
    return np.asarray(new_pairs, dtype=np.int64).reshape(-1, 2)

print("Eval helpers ready.")

Eval helpers ready.


In [11]:
# ============================================================
# 11. Train STEA (iterative)
# ============================================================

def train_one_iteration(model, optimizer, train_pairs, epochs=1200):
    model.train()
    train_pairs_np = np.asarray(train_pairs, dtype=np.int64)
    history = []
    for epoch in range(1, epochs + 1):
        optimizer.zero_grad()
        emb = model(adj)
        perm = np.random.permutation(len(train_pairs_np))
        loss = stea_triplet_loss(emb, train_pairs_np[perm], margin=MARGIN)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        history.append(float(loss.detach().cpu()))
        if epoch == 1 or epoch % 100 == 0 or epoch == epochs:
            print(f"Epoch {epoch:4d}/{epochs} | loss={history[-1]:.6f}")
    return history


current_train_pairs = seed_pairs.copy()
all_histories = []
n_iterations = ITERATIONS if USE_ITERATIVE_STEA else 1

for iteration in range(1, n_iterations + 1):
    print("\n" + "=" * 70)
    print(f"STEA iteration {iteration}/{n_iterations}")
    print("Training pairs:", len(current_train_pairs))
    print("=" * 70)

    history = train_one_iteration(
        model, optimizer, current_train_pairs, epochs=EPOCHS_PER_ITER
    )
    all_histories.append(history)

    model.eval()
    with torch.no_grad():
        final_emb = model(adj)

    result = evaluate_embeddings(
        final_emb, eval_pairs, alpha=ALPHA, use_csls=True, csls_k=CSLS_K
    )
    print_metrics(result, f"Evaluation after iteration {iteration}")

    if not USE_ITERATIVE_STEA:
        break

    new_pairs = mutual_nearest_pairs(result["similarity"], existing_pairs=current_train_pairs)
    print("New mutual-NN pairs:", len(new_pairs))
    if len(new_pairs) == 0:
        print("No new pairs; stopping.")
        break
    current_train_pairs = np.vstack([current_train_pairs, new_pairs])

print("\nFinal training-pair count:", len(current_train_pairs))


STEA iteration 1/5
Training pairs: 200
Epoch    1/1200 | loss=0.457869
Epoch  100/1200 | loss=0.045484
Epoch  200/1200 | loss=0.000000
Epoch  300/1200 | loss=0.001935
Epoch  400/1200 | loss=0.010741
Epoch  500/1200 | loss=0.000000
Epoch  600/1200 | loss=0.000000
Epoch  700/1200 | loss=0.000000
Epoch  800/1200 | loss=0.000000
Epoch  900/1200 | loss=0.000000
Epoch 1000/1200 | loss=0.000000
Epoch 1100/1200 | loss=0.000000
Epoch 1200/1200 | loss=0.000000

Evaluation after iteration 1
----------------------------
left_to_right
  Hits@1  : 72.24%
  Hits@5  : 84.12%
  Hits@10 : 87.30%
  MRR     : 0.7788
right_to_left
  Hits@1  : 70.00%
  Hits@5  : 82.70%
  Hits@10 : 85.72%
  MRR     : 0.7584
New mutual-NN pairs: 4455

STEA iteration 2/5
Training pairs: 4655
Epoch    1/1200 | loss=0.193861
Epoch  100/1200 | loss=0.001004
Epoch  200/1200 | loss=0.005229
Epoch  300/1200 | loss=0.001713
Epoch  400/1200 | loss=0.002067
Epoch  500/1200 | loss=0.000159
Epoch  600/1200 | loss=0.001985
Epoch  700/120

In [12]:
# ============================================================
# 12. Final evaluation
# ============================================================

model.eval()
with torch.no_grad():
    final_embeddings = model(adj)

final_result = evaluate_embeddings(
    final_embeddings, eval_pairs, alpha=ALPHA, use_csls=True, csls_k=CSLS_K
)
print_metrics(final_result, "FINAL STEA RESULT")

print("\nConfiguration")
print("-------------")
print("LANG_PAIR:", LANG_PAIR)
print("TRAIN_SEED:", TRAIN_SEED)
print("DIM:", DIM)
print("DEPTH:", DEPTH)
print("MARGIN:", MARGIN)
print("DROPOUT:", DROPOUT)
print("ALPHA:", ALPHA)
print("LR:", LR)
print("EPOCHS_PER_ITER:", EPOCHS_PER_ITER)
print("ITERATIONS:", n_iterations)
print("CSLS_K:", CSLS_K)


FINAL STEA RESULT
-----------------
left_to_right
  Hits@1  : 83.73%
  Hits@5  : 91.32%
  Hits@10 : 93.38%
  MRR     : 0.8732
right_to_left
  Hits@1  : 79.43%
  Hits@5  : 89.59%
  Hits@10 : 91.76%
  MRR     : 0.8407

Configuration
-------------
LANG_PAIR: en_fa
TRAIN_SEED: 200
DIM: 100
DEPTH: 2
MARGIN: 3.0
DROPOUT: 0.3
ALPHA: 0.3
LR: 0.005
EPOCHS_PER_ITER: 1200
ITERATIONS: 5
CSLS_K: 10


In [13]:
# ============================================================
# 13. Save checkpoint
# ============================================================

CHECKPOINT_PATH = Path(f"/content/drive/MyDrive/stea_wikidata12k_{LANG_PAIR}_checkpoint.pt")

torch.save({
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
    "config": {
        "LANG_PAIR": LANG_PAIR,
        "TRAIN_SEED": TRAIN_SEED,
        "DIM": DIM,
        "DEPTH": DEPTH,
        "MARGIN": MARGIN,
        "DROPOUT": DROPOUT,
        "ALPHA": ALPHA,
        "LR": LR,
        "EPOCHS_PER_ITER": EPOCHS_PER_ITER,
        "ITERATIONS": ITERATIONS,
        "CSLS_K": CSLS_K,
        "N1": N1,
        "N2": N2,
    },
    "current_train_pairs": current_train_pairs,
    "final_metrics": {
        "left_to_right": final_result["left_to_right"],
        "right_to_left": final_result["right_to_left"],
    },
}, CHECKPOINT_PATH)

print("Saved:", CHECKPOINT_PATH)

Saved: /content/drive/MyDrive/stea_wikidata12k_en_fa_checkpoint.pt


## Notes

- **Timestamps** in this dataset are year-like strings (`1926-##-##`). They are converted to integer years for STEA’s time dictionary.
- **Entity IDs** are local per language; the notebook remaps them into disjoint namespaces (KG2 offset by N1).
- **Relations** share one vocabulary (`relation2id.tsv`) across both KGs.
- Change `LANG_PAIR` to `en_fa`, `en_zh`, or `en_ja` and re-run from the config cell.
- For fair method comparison, keep the same `TRAIN_SEED` and evaluation pool protocol across RREA / TEA-GNN / TREA / STEA.